# Notebook to Compare Heuristics

In [31]:
import asyncio
import nest_asyncio

import pandas as pd
from time import time

import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

In [32]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)

In [33]:
nest_asyncio.apply()

In [34]:
from src.heuristics import (
    welfare_greedy,
    c_fim,
    kempe_greedy,
)

from src.diffusion_models import (
    estimate_cascade_influence,
    estimate_cascade_by_community,
)
from src.metrics import utility_gap

In [35]:
from src import Loader
from pathlib import Path

path_to_networks = Path('../data/synthetic/networks/')
path_to_results = Path('../../results/barbasi_albert/size_200/')

file_name = 'barbasi_albert_200'

In [36]:
async def main():
    loader = Loader(max_workers=4)
    graph = await loader.load(f'{path_to_networks}/{file_name}.pkl')
    communities = list(greedy_modularity_communities(graph))
    costs = nx.get_node_attributes(graph, 'node_costs')

    return graph, communities, costs


graph, communities, costs = asyncio.run(main())

## Comparison of Welfare Greedy and Kempe Greedy

In [37]:
alpha_fixed = 0
p_fixed = 0.1
k_values = [1, 5, 10, 20]
num_sims = 1000

In [38]:
results_k_variation = []

for k in k_values:
    random.seed(42)
    np.random.seed(42)
    # Welfare-based approach
    start = time()
    welfare_seeds = welfare_greedy(
        graph=graph,
        communities=communities,
        k=k,
        alpha=alpha_fixed,
        probability=p_fixed,
        num_sims=num_sims,
    )
    welfare_time = time() - start

    welfare_influence = estimate_cascade_influence(
        graph=graph,
        seeds=welfare_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    welfare_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=welfare_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    welfare_by_comm_rounded = {comm: round(val, 2) for comm, val in welfare_by_comm.items()}

    # Kempe et al. original greedy approach
    start = time()
    kempe_seeds = kempe_greedy(
        graph=graph,
        k=k,
        probability=p_fixed,
        num_simulations=num_sims,
    )
    kempe_time = time() - start

    kempe_influence = estimate_cascade_influence(
        graph=graph,
        seeds=kempe_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    kempe_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=kempe_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    kempe_by_comm_rounded = {comm: round(val, 2) for comm, val in kempe_by_comm.items()}

    # Calculate utility gap and PoF
    utility_gap = max(welfare_by_comm.values()) - min(welfare_by_comm.values())

    welfare_ugap = max(welfare_by_comm.values()) - min(welfare_by_comm.values())
    kempe_ugap = max(kempe_by_comm.values()) - min(kempe_by_comm.values())

    pof = 1 - (welfare_influence / kempe_influence) if kempe_influence > 0 else 0

    results_k_variation.append(
        {
            'k': k,
            'alpha': alpha_fixed,
            'p': p_fixed,
            'kempe_seeds': kempe_seeds,
            'kempe_time_s': kempe_time,
            'kempe_total_influence': kempe_influence,
            'kempe_influence_by_community': kempe_by_comm_rounded,
            'kempe_utility_gap': kempe_ugap,
            'welfare_seeds': welfare_seeds,
            'welfare_time_s': welfare_time,
            'welfare_total_influence': welfare_influence,
            'welfare_influence_by_community': welfare_by_comm_rounded,
            'welfare_utility_gap': welfare_ugap,
            'price_of_fairness': pof,
        }
    )

Selecting seeds: 100%|██████████| 20/20 [00:05<00:00,  3.41it/s, seeds=20]


In [39]:
df_k_variation = pd.DataFrame(results_k_variation)
df_k_variation.to_csv(f'{path_to_results}/kempe_welfare_k_variation_{file_name}.csv', index=False)

In [40]:
k_fixed = 10  # fixed number of seeds
p_fixed = 0.1  # edge activation probability
alpha_values = [0.5, 0.0, -2.0, -5.0, -7.0, -9.0]  # varying inequality-aversion parameter
num_sims = 1000

In [41]:
results_alpha_variation = []

for alpha in alpha_values:
    random.seed(42)
    np.random.seed(42)
    start = time()
    kempe_seeds = kempe_greedy(
        graph=graph,
        k=k_fixed,
        probability=p_fixed,
        num_simulations=num_sims,
    )
    kempe_time = time() - start

    kempe_influence = estimate_cascade_influence(
        graph=graph,
        seeds=kempe_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    kempe_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=kempe_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    kempe_by_comm_rounded = {comm: round(val, 2) for comm, val in kempe_by_comm.items()}

    # Welfare-based approach with varying alpha
    start = time()
    welfare_seeds = welfare_greedy(
        graph=graph,
        communities=communities,
        k=k_fixed,
        alpha=alpha,
        probability=p_fixed,
        num_sims=num_sims,
    )
    welfare_time = time() - start

    welfare_influence = estimate_cascade_influence(
        graph=graph,
        seeds=welfare_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    welfare_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=welfare_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    welfare_by_comm_rounded = {comm: round(val, 2) for comm, val in welfare_by_comm.items()}

    # Calculate utility gap and PoF
    utility_gap = max(welfare_by_comm.values()) - min(welfare_by_comm.values())

    welfare_ugap = max(welfare_by_comm.values()) - min(welfare_by_comm.values())
    kempe_ugap = max(kempe_by_comm.values()) - min(kempe_by_comm.values())

    pof = 1 - (welfare_influence / kempe_influence) if kempe_influence > 0 else 0

    results_alpha_variation.append(
        {
            'k': k,
            'alpha': alpha,
            'p': p_fixed,
            'kempe_seeds': kempe_seeds,
            'kempe_time_s': kempe_time,
            'kempe_total_influence': kempe_influence,
            'kempe_influence_by_community': kempe_by_comm_rounded,
            'kempe_utility_gap': kempe_ugap,
            'welfare_seeds': welfare_seeds,
            'welfare_time_s': welfare_time,
            'welfare_total_influence': welfare_influence,
            'welfare_influence_by_community': welfare_by_comm_rounded,
            'welfare_utility_gap': welfare_ugap,
            'price_of_fairness': pof,
        }
    )

Selecting seeds: 100%|██████████| 10/10 [00:02<00:00,  3.65it/s, seeds=10, influenced={0: 0.09, 1: 0.1, 2: 0.09, 3: 0.11, 4: 0.1, 5: 0.12, 6: 0.11, 7: 0.1, 8: 0.1, 9: 0.08}] 


In [42]:
df_alpha_variation = pd.DataFrame(results_alpha_variation)
df_alpha_variation.to_csv(f'{path_to_results}/kempe_welfare_alpha_variation_{file_name}.csv', index=False)

## Comparison of Welfare Greedy and Budgeted Welfare Greedy (CFIM)

In [43]:
k_fixed = 10  # number of seeds to select
p_fixed = 0.1  # edge activation probability
budget_fixed = 15.0  # Total budget available
alphas = [0.5, 0.0, -2.0, -5.0, -7.0, -9.0]  # varying inequality-aversion parameter
num_sims = 1000

In [44]:
results_welfare_cfim = []

for alpha in alphas:
    random.seed(42)
    np.random.seed(42)
    # Welfare-based approach
    start = time()
    welfare_seeds = welfare_greedy(
        graph=graph,
        communities=communities,
        k=k_fixed,
        alpha=alpha,
        probability=p_fixed,
        num_sims=num_sims,
    )
    welfare_time = time() - start

    welfare_influence = estimate_cascade_influence(
        graph=graph,
        seeds=welfare_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    welfare_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=welfare_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    welfare_by_comm_rounded = {comm: round(val, 2) for comm, val in welfare_by_comm.items()}
    welfare_ugap = max(welfare_by_comm.values()) - min(welfare_by_comm.values())

    # C-FIM approach
    start = time()
    cfim_seeds = c_fim(
        graph=graph,
        communities=communities,
        budget=budget_fixed,
        costs=costs,
        alpha=alpha,
        probability=p_fixed,
        num_sims=num_sims,
    )
    cfim_time = time() - start

    cfim_influence = estimate_cascade_influence(
        graph=graph,
        seeds=cfim_seeds,
        probability=p_fixed,
        num_simulations=num_sims,
        random_state=42,
    )

    cfim_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=cfim_seeds,
        probability=p_fixed,
        num_simulations=num_sims // 2,
        random_state=42,
    )
    cfim_by_comm_rounded = {comm: round(val, 2) for comm, val in cfim_by_comm.items()}
    cfim_ugap = max(cfim_by_comm.values()) - min(cfim_by_comm.values())

    # Fairness metrics
    price_of_fairness = 1 - (welfare_influence / cfim_influence) if cfim_influence > 0 else 0
    cfim_advantage = cfim_influence > welfare_influence

    # Seed counts (may differ with budgeted C-FIM)
    welfare_seed_count = len(welfare_seeds) if isinstance(welfare_seeds, (list, set)) else k_fixed
    cfim_seed_count = len(cfim_seeds) if isinstance(cfim_seeds, (list, set)) else len(cfim_seeds)

    results_welfare_cfim.append(
        {
            'alpha': alpha,
            'k': k_fixed,
            'p': p_fixed,
            'budget': budget_fixed,

            # Welfare-based results
            'welfare_seeds': welfare_seeds,
            'welfare_seed_count': welfare_seed_count,
            'welfare_time_s': welfare_time,
            'welfare_total_influence': welfare_influence,
            'welfare_influence_by_community': welfare_by_comm_rounded,
            'utility_gap_welfare': welfare_ugap,

            # C-FIM results
            'cfim_seeds': cfim_seeds,
            'cfim_seed_count': cfim_seed_count,
            'cfim_time_s': cfim_time,
            'cfim_total_influence': cfim_influence,
            'cfim_influence_by_community': cfim_by_comm_rounded,
            'utility_gap_cfim': cfim_ugap,

            # Comparison metrics
            'price_of_fairness': price_of_fairness,
            'cfim_advantage': cfim_advantage,
            'total_execution_time_s': welfare_time + cfim_time,
            'time_ratio_cfim_vs_welfare': cfim_time / welfare_time if welfare_time > 0 else float('inf'),
        }
    )

No more affordable candidates:   6%|▌         | 11/200 [00:00<00:15, 11.91it/s, seeds=11, cost=14.40/15.00, remaining_budget=0.60, influenced={0: 0.14, 1: 0.22, 2: 0.05, 3: 0.06, 4: 0.1, 5: 0.12, 6: 0.08, 7: 0.08, 8: 0.06, 9: 0.11}]


In [45]:
df_welfare_cfim = pd.DataFrame(results_welfare_cfim)
output_filename = f'{path_to_results}/welfare_cfim_alpha_comparison_{file_name}.csv'
df_welfare_cfim.to_csv(output_filename, index=False)